# Entraînement + évaluation sur Kaggle (GPU T4)
**Avant de lancer** (panneau de droite, *Session options*) :
- *Accelerator* : **GPU T4 x2** (ou P100)
- *Internet* : **On** (il faut avoir vérifié son numéro de téléphone sur Kaggle)
- *Input* : ajouter le dataset `portes-marocaines-data` (dossier `raw/` + `sources.csv` + `captions.csv` corrigé)

Puis **Save Version > Save & Run All (Commit)** : le calcul continue même navigateur fermé (≈ 5 h).
À la fin : onglet *Output* > télécharger `resultats.zip`.

In [ ]:
!git clone -q https://github.com/oussioumaima7-star/lora-portes-marocaines.git /kaggle/working/projet
%cd /kaggle/working/projet
!pip install -q diffusers peft transformers accelerate safetensors
import torch; print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'Go')

## Données
On cherche automatiquement le dataset qui contient `sources.csv`.

In [ ]:
import glob, os, shutil, pandas as pd
src_csv = glob.glob('/kaggle/input/**/sources.csv', recursive=True)
assert src_csv, 'Dataset introuvable : ajouter le dataset en Input'
D = os.path.dirname(src_csv[0]); print('Dataset :', D)
os.makedirs('data/raw', exist_ok=True)
raw = os.path.join(D, 'raw') if os.path.isdir(os.path.join(D, 'raw')) else D
for f in glob.glob(raw + '/*'):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')): shutil.copy(f, 'data/raw/')
for f in ['sources.csv', 'captions.csv']:
    if os.path.exists(os.path.join(D, f)): shutil.copy(os.path.join(D, f), 'data/' + f)
print(len(os.listdir('data/raw')), 'images')

In [ ]:
# Même découpage qu'avant (graine fixe) ; les légendes corrigées sont reprises par nom de source
!python data/prepare.py --heldout 8 --mode crop
cap = pd.read_csv('data/captions.csv')
todo = cap[cap.caption.str.contains('TODO')]
assert todo.empty, f'{len(todo)} légendes non remplies : corriger captions.csv'
cap.head()

## Entraînements (≈ 4 h)

In [ ]:
import subprocess, os
runs = {'rank8': 'rank8', 'rank4': 'rank4', 'rank16': 'rank16', 'rank32': 'rank32',
        'steps_long': 'rank8_steps2000'}
for cfg, name in runs.items():
    if os.path.exists(f'results/loras/{name}/final'):
        print(cfg, 'déjà fait'); continue
    print('=====', cfg, flush=True)
    subprocess.run(['python', '-m', 'src.train_lora', '--config', f'configs/{cfg}.yaml'], check=True)

## Évaluation, figures, résumé (≈ 1 h)

In [ ]:
!python -m src.evaluate --loras results/loras/rank4/final results/loras/rank8/final results/loras/rank16/final results/loras/rank32/final
!cp results/eval/metrics.csv results/eval/metrics_rangs.csv
!python -m src.evaluate --quick --out results/eval_etapes --loras results/loras/rank8_steps2000/checkpoint-* results/loras/rank8_steps2000/final
a = pd.read_csv('results/eval/metrics_rangs.csv'); b = pd.read_csv('results/eval_etapes/metrics.csv')
pd.concat([a, b[b.lora != 'base']]).to_csv('results/eval/metrics.csv', index=False)
!python -m src.plots
!python -m src.summary
print(open('results/resume.md').read())

## Export
Les LoRA finales (quelques Mo chacune) + figures + grilles + métriques. Les images générées individuelles restent hors du zip.

In [ ]:
!cd /kaggle/working/projet && zip -qr /kaggle/working/resultats.zip results/resume.md results/figures results/eval/*.csv results/eval/*.jpg results/eval_etapes/*.csv results/eval_etapes/*.jpg results/loras/*/final results/loras/*/*.json results/loras/*/*.csv data/captions.csv
!ls -lh /kaggle/working/resultats.zip
!rm -rf /kaggle/working/projet/results/eval/*/  /kaggle/working/projet/results/eval_etapes/*/  # libère l'espace Output